# Notebook 08 — Clasificación Factorial de Imágenes

**Proyecto:** Modelo Cognitivo Artificial para la Replicación de la Actividad Neurofisiológica de Percepción de Profundidad  
**Autor:** Jesús Goenaga Peña  
**Fase:** 1 del pipeline de validación — Clasificación factorial

---

## ¿Qué hace este notebook?

Este notebook toma las imágenes del conjunto de validación (las 200 que se usarán con participantes humanos) y las clasifica según el diseño factorial 2×2×2 del estudio:

- **Factor 1:** Tipo de tarea → KITTI (discriminación real) vs. Ilusiones visuales
- **Factor 2:** Nivel de disparidad binocular → Alta vs. Baja
- **Factor 3:** Presencia de distractores visuales → Con vs. Sin

El resultado es un archivo CSV maestro (`factorial_labels.csv`) que asigna a cada imagen su condición factorial. Este CSV es la base de todo el análisis posterior.

**Criterios técnicos:**
- Disparidad: calculada desde los mapas ground truth de KITTI. Umbral = mediana de la distribución.
- Distractores: densidad de bordes calculada con el operador Sobel. Umbral = mediana de la distribución.

---

## Celda 1 — Montar Google Drive y verificar rutas

In [ ]:
# ============================================================
# CELDA 1: Montar Drive y verificar estructura de carpetas
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os

# Ruta base del proyecto en tu Drive
BASE_DIR = '/content/drive/MyDrive/cognitive-depth-model'

# Rutas donde viven los datos de KITTI
# Los mapas de disparidad ground truth están en disp_occ
KITTI_DISP_DIR   = os.path.join(BASE_DIR, 'data', 'kitti', 'training', 'disp_occ_0')
KITTI_LEFT_DIR   = os.path.join(BASE_DIR, 'data', 'kitti', 'training', 'image_2')

# Ruta donde viven las imágenes del dataset de ilusiones visuales
ILLUSION_DIR     = os.path.join(BASE_DIR, 'data', 'illusion')

# Ruta de splits (CSVs generados por notebooks anteriores)
SPLITS_DIR       = os.path.join(BASE_DIR, 'data', 'splits')

# Ruta de salida para este notebook
RESULTS_DIR      = os.path.join(BASE_DIR, 'results', 'factorial')
os.makedirs(RESULTS_DIR, exist_ok=True)

# Verificar que las rutas principales existen
rutas = {
    'KITTI disparidad': KITTI_DISP_DIR,
    'KITTI imágenes': KITTI_LEFT_DIR,
    'Ilusiones': ILLUSION_DIR,
    'Splits': SPLITS_DIR,
    'Resultados': RESULTS_DIR
}

print('Verificación de rutas:')
print('-' * 50)
for nombre, ruta in rutas.items():
    estado = '✓ Existe' if os.path.exists(ruta) else '✗ NO ENCONTRADA'
    print(f'  {nombre}: {estado}')
    print(f'    {ruta}')

print()
print('IMPORTANTE: Si alguna ruta dice NO ENCONTRADA, detente aquí')
print('y escríbeme el mensaje de error para corregir la ruta antes de continuar.')

## Celda 2 — Instalar dependencias e importar librerías

In [ ]:
# ============================================================
# CELDA 2: Importar librerías
# ============================================================
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print('Librerías importadas correctamente.')
print(f'  NumPy: {np.__version__}')
print(f'  OpenCV: {cv2.__version__}')
print(f'  Pandas: {pd.__version__}')

## Celda 3 — Cargar el listado de imágenes de validación

Aquí cargamos los nombres de las imágenes que ya fueron reservadas para validación en los notebooks anteriores. Si existe un CSV de splits previo, lo usamos. Si no, construimos el listado directamente desde la carpeta.

In [ ]:
# ============================================================
# CELDA 3: Cargar listado de imágenes de validación
# ============================================================

# --- KITTI: imágenes de validación ---
# Intentamos cargar desde el CSV de splits existente
kitti_split_path = os.path.join(SPLITS_DIR, 'kitti_splits.csv')

if os.path.exists(kitti_split_path):
    df_kitti_split = pd.read_csv(kitti_split_path)
    # Filtramos las que pertenecen al conjunto de validación
    kitti_val_ids = df_kitti_split[
        df_kitti_split['split'] == 'validation'
    ]['image_id'].tolist()
    print(f'CSV de splits KITTI cargado. Imágenes de validación: {len(kitti_val_ids)}')
else:
    # Si no hay CSV previo, tomamos las primeras 100 disponibles en disp_occ
    # (las usadas en entrenamiento/prueba deberían estar documentadas en notebooks anteriores)
    print('No se encontró CSV de splits KITTI. Usando todas las disponibles en disp_occ.')
    kitti_val_ids = sorted([
        f.stem for f in Path(KITTI_DISP_DIR).glob('*.png')
    ])[:100]  # Tomamos 100 para validación
    print(f'Imágenes KITTI disponibles para validación: {len(kitti_val_ids)}')

print(f'Ejemplo de IDs KITTI: {kitti_val_ids[:5]}')
print()

# --- Ilusiones: imágenes de validación ---
illusion_split_path = os.path.join(SPLITS_DIR, 'illusion_splits.csv')

if os.path.exists(illusion_split_path):
    df_illusion_split = pd.read_csv(illusion_split_path)
    illusion_val_ids = df_illusion_split[
        df_illusion_split['split'] == 'validation'
    ]['image_id'].tolist()
    print(f'CSV de splits Ilusiones cargado. Imágenes de validación: {len(illusion_val_ids)}')
else:
    print('No se encontró CSV de splits Ilusiones. Usando imágenes disponibles.')
    illusion_files = list(Path(ILLUSION_DIR).rglob('*.png'))[:100]
    illusion_files += list(Path(ILLUSION_DIR).rglob('*.jpg'))[:100]
    illusion_val_ids = [f.stem for f in illusion_files[:100]]
    print(f'Imágenes de ilusiones disponibles para validación: {len(illusion_val_ids)}')

print(f'Ejemplo de IDs ilusiones: {illusion_val_ids[:5]}')
print()
print(f'Total imágenes a clasificar: {len(kitti_val_ids) + len(illusion_val_ids)}')

## Celda 4 — Funciones de clasificación factorial

Aquí definimos las dos funciones clave:
- `calcular_disparidad_media()`: lee el mapa PNG de KITTI y calcula la disparidad promedio de los píxeles válidos
- `calcular_densidad_bordes()`: calcula la complejidad visual de una imagen como proxy de distractores

In [ ]:
# ============================================================
# CELDA 4: Definir funciones de clasificación
# ============================================================

def calcular_disparidad_media(ruta_png):
    """
    Lee un mapa de disparidad KITTI (PNG 16-bit) y devuelve
    la disparidad media sobre píxeles válidos.
    
    En KITTI, el valor PNG se divide entre 256 para obtener
    la disparidad real en píxeles. Los píxeles con valor 0
    son inválidos (sin dato LiDAR) y se excluyen.
    
    Retorna: float (disparidad media) o np.nan si no hay datos.
    """
    mapa = cv2.imread(str(ruta_png), cv2.IMREAD_UNCHANGED)
    if mapa is None:
        return np.nan
    
    # Convertir a float y aplicar escala KITTI
    mapa_real = mapa.astype(np.float32) / 256.0
    
    # Máscara de píxeles válidos (valor original > 0)
    pixeles_validos = mapa_real[mapa_real > 0]
    
    if len(pixeles_validos) == 0:
        return np.nan
    
    return float(np.mean(pixeles_validos))


def calcular_densidad_bordes(ruta_imagen):
    """
    Calcula la densidad de bordes de una imagen como medida
    de complejidad visual (proxy de distractores).
    
    Usa el operador Sobel para detectar gradientes de intensidad.
    La densidad se normaliza entre 0 y 1 dividiendo entre el
    máximo teórico, lo que la hace comparable entre imágenes
    de diferente resolución.
    
    Retorna: float (densidad de bordes normalizada, 0-1)
    """
    img = cv2.imread(str(ruta_imagen), cv2.IMREAD_GRAYSCALE)
    if img is None:
        return np.nan
    
    # Redimensionar a tamaño estándar para comparabilidad
    img = cv2.resize(img, (640, 192))
    
    # Suavizado previo para reducir ruido de alta frecuencia
    img_suave = cv2.GaussianBlur(img, (3, 3), 0)
    
    # Gradientes Sobel en X e Y
    grad_x = cv2.Sobel(img_suave, cv2.CV_64F, 1, 0, ksize=3)
    grad_y = cv2.Sobel(img_suave, cv2.CV_64F, 0, 1, ksize=3)
    
    # Magnitud del gradiente
    magnitud = np.sqrt(grad_x**2 + grad_y**2)
    
    # Densidad normalizada: promedio sobre máximo teórico
    # El máximo teórico de Sobel 3x3 sobre imagen uint8 es ~1442
    densidad = float(np.mean(magnitud) / 1442.0)
    
    return densidad


print('Funciones definidas correctamente.')
print()
print('Resumen de criterios:')
print('  - Disparidad: mediana de la distribución como umbral (Alta ≥ mediana, Baja < mediana)')
print('  - Distractores: mediana de la distribución como umbral (Con ≥ mediana, Sin < mediana)')

## Celda 5 — Clasificar imágenes KITTI

In [ ]:
# ============================================================
# CELDA 5: Clasificar imágenes KITTI
# ============================================================
print('Procesando imágenes KITTI...')
print('Esto puede tardar 1-2 minutos.\n')

registros_kitti = []

for img_id in tqdm(kitti_val_ids, desc='KITTI'):
    # Ruta al mapa de disparidad
    ruta_disp = Path(KITTI_DISP_DIR) / f'{img_id}.png'
    # Ruta a la imagen de color (ojo izquierdo)
    ruta_img  = Path(KITTI_LEFT_DIR) / f'{img_id}.png'
    
    # Calcular métricas
    disp_media   = calcular_disparidad_media(ruta_disp) if ruta_disp.exists() else np.nan
    densidad_brd = calcular_densidad_bordes(ruta_img)   if ruta_img.exists()  else np.nan
    
    registros_kitti.append({
        'imagen_id':        img_id,
        'dataset':          'KITTI',
        'tipo_tarea':       'discriminacion_profundidad',
        'ruta_img_izq':     str(ruta_img),
        'ruta_disp':        str(ruta_disp),
        'disparidad_media': disp_media,
        'densidad_bordes':  densidad_brd
    })

df_kitti = pd.DataFrame(registros_kitti)

# Reporte de datos faltantes
nan_disp = df_kitti['disparidad_media'].isna().sum()
nan_brd  = df_kitti['densidad_bordes'].isna().sum()
print(f'\nTotal imágenes KITTI procesadas: {len(df_kitti)}')
print(f'  Mapas de disparidad no encontrados: {nan_disp}')
print(f'  Imágenes de color no encontradas:   {nan_brd}')
print()

# Estadísticas descriptivas de las métricas
print('Estadísticas de disparidad media (KITTI):')
print(df_kitti['disparidad_media'].describe().round(4))
print()
print('Estadísticas de densidad de bordes (KITTI):')
print(df_kitti['densidad_bordes'].describe().round(6))

## Celda 6 — Clasificar imágenes de Ilusiones Visuales

In [ ]:
# ============================================================
# CELDA 6: Clasificar imágenes de Ilusiones Visuales
# ============================================================
# Para el dataset de ilusiones NO hay mapas de disparidad ground truth
# en el mismo formato KITTI. En cambio, el dataset incluye mapas de
# profundidad que convertimos a disparidad equivalente.
# Si no hay mapa de profundidad disponible, usamos la densidad de bordes
# como única métrica y la disparidad queda como NaN (se resolverá más adelante).

print('Procesando imágenes de Ilusiones Visuales...')
print('Esto puede tardar 1-2 minutos.\n')

# Buscar imágenes recursivamente dentro de ILLUSION_DIR
# El dataset tiene subcarpetas por tipo de ilusión
illusion_paths = {}
for ext in ['*.png', '*.jpg', '*.jpeg']:
    for ruta in Path(ILLUSION_DIR).rglob(ext):
        illusion_paths[ruta.stem] = ruta

print(f'Total de archivos de ilusiones encontrados: {len(illusion_paths)}')

registros_illusion = []

for img_id in tqdm(illusion_val_ids, desc='Ilusiones'):
    ruta_img = illusion_paths.get(img_id, None)
    
    if ruta_img is None:
        # Intentar búsqueda parcial por si el ID no coincide exactamente
        coincidencias = [v for k, v in illusion_paths.items() if img_id in k]
        ruta_img = coincidencias[0] if coincidencias else None
    
    densidad_brd = calcular_densidad_bordes(ruta_img) if ruta_img else np.nan
    
    # Determinar tipo de ilusión desde la ruta (subcarpeta del dataset)
    tipo_ilusion = ruta_img.parent.name if ruta_img else 'desconocido'
    
    registros_illusion.append({
        'imagen_id':        img_id,
        'dataset':          '3D_Illusion',
        'tipo_tarea':       'ilusion_visual',
        'tipo_ilusion':     tipo_ilusion,
        'ruta_img_izq':     str(ruta_img) if ruta_img else '',
        'ruta_disp':        '',  # No aplica para ilusiones
        'disparidad_media': np.nan,  # Se completará en celda siguiente
        'densidad_bordes':  densidad_brd
    })

df_illusion = pd.DataFrame(registros_illusion)

print(f'\nTotal imágenes de ilusiones procesadas: {len(df_illusion)}')
print(f'  Tipos de ilusión encontrados: {df_illusion["tipo_ilusion"].unique().tolist()}')
print()
print('Estadísticas de densidad de bordes (Ilusiones):')
print(df_illusion['densidad_bordes'].describe().round(6))

## Celda 7 — Calcular umbrales y asignar etiquetas factoriales

In [ ]:
# ============================================================
# CELDA 7: Calcular umbrales y asignar etiquetas factoriales
# ============================================================

# --- PASO 1: Combinar ambos datasets ---
df_total = pd.concat([df_kitti, df_illusion], ignore_index=True)
print(f'Total imágenes combinadas: {len(df_total)}')
print(f'  KITTI: {len(df_kitti)}')
print(f'  Ilusiones: {len(df_illusion)}')
print()

# --- PASO 2: Umbral de disparidad (solo sobre KITTI que tiene datos reales) ---
# Para ilusiones, la disparidad se clasifica internamente dentro de su subgrupo
# usando la densidad de bordes como proxy (ver paso 4)
disp_validas = df_total['disparidad_media'].dropna()
UMBRAL_DISPARIDAD = disp_validas.median()
print(f'Umbral de disparidad (mediana KITTI): {UMBRAL_DISPARIDAD:.4f} píxeles')

# --- PASO 3: Umbral de distractores (sobre todos los datasets combinados) ---
brd_validas = df_total['densidad_bordes'].dropna()
UMBRAL_BORDES = brd_validas.median()
print(f'Umbral de distractores (mediana global): {UMBRAL_BORDES:.6f}')
print()

# --- PASO 4: Asignar etiqueta de disparidad ---
def etiquetar_disparidad(row):
    if row['dataset'] == 'KITTI':
        # KITTI: usa el mapa de disparidad real
        if pd.isna(row['disparidad_media']):
            return 'sin_dato'
        return 'alta' if row['disparidad_media'] >= UMBRAL_DISPARIDAD else 'baja'
    else:
        # Ilusiones: usa densidad de bordes como proxy de "fuerza" de la ilusión
        # Mayor complejidad visual → ilusión más fuerte → mayor disparidad percibida
        if pd.isna(row['densidad_bordes']):
            return 'sin_dato'
        umbral_ilusion = df_illusion['densidad_bordes'].median()
        return 'alta' if row['densidad_bordes'] >= umbral_ilusion else 'baja'

df_total['nivel_disparidad'] = df_total.apply(etiquetar_disparidad, axis=1)

# --- PASO 5: Asignar etiqueta de distractores ---
def etiquetar_distractores(row):
    if pd.isna(row['densidad_bordes']):
        return 'sin_dato'
    return 'con_distractores' if row['densidad_bordes'] >= UMBRAL_BORDES else 'sin_distractores'

df_total['presencia_distractores'] = df_total.apply(etiquetar_distractores, axis=1)

# --- PASO 6: Etiqueta combinada de condición factorial ---
df_total['condicion_factorial'] = (
    df_total['tipo_tarea'].str[:4].str.upper() + '_' +
    df_total['nivel_disparidad'].str[0].str.upper() + 'disp_' +
    df_total['presencia_distractores'].map({
        'con_distractores': 'ConDist',
        'sin_distractores': 'SinDist',
        'sin_dato': 'SinDato'
    })
)

print('Distribución factorial resultante:')
print('='*60)
print(df_total.groupby(['dataset', 'nivel_disparidad', 'presencia_distractores']).size().to_string())
print()

# Verificar balance del diseño
print('Conteo por celda factorial (ideal: 25 por celda):')
tabla = df_total.groupby(
    ['tipo_tarea', 'nivel_disparidad', 'presencia_distractores']
).size().reset_index(name='n_imagenes')
print(tabla.to_string(index=False))

## Celda 8 — Visualizaciones de las distribuciones

In [ ]:
# ============================================================
# CELDA 8: Visualizaciones de las distribuciones
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(
    'Clasificación Factorial — Distribuciones de Disparidad y Complejidad Visual',
    fontsize=14, fontweight='bold', y=1.02
)

# --- GRÁFICA 1: Distribución de disparidad media (KITTI) ---
ax1 = axes[0, 0]
kitti_disp = df_total[df_total['dataset'] == 'KITTI']['disparidad_media'].dropna()
colores_disp = ['#2196F3' if v >= UMBRAL_DISPARIDAD else '#FF9800' for v in kitti_disp]
ax1.hist(kitti_disp, bins=25, color='#90CAF9', edgecolor='white', alpha=0.8)
ax1.axvline(UMBRAL_DISPARIDAD, color='#D32F2F', linewidth=2, linestyle='--',
            label=f'Mediana = {UMBRAL_DISPARIDAD:.2f} px')
ax1.set_xlabel('Disparidad media por escena (píxeles)')
ax1.set_ylabel('Número de escenas')
ax1.set_title('Distribución de disparidad — KITTI')
ax1.legend()
ax1.text(0.05, 0.92, f'Alta disparidad (≥ umbral): {(kitti_disp >= UMBRAL_DISPARIDAD).sum()}\n'
         f'Baja disparidad (< umbral): {(kitti_disp < UMBRAL_DISPARIDAD).sum()}',
         transform=ax1.transAxes, fontsize=9,
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

# --- GRÁFICA 2: Distribución de densidad de bordes (todos) ---
ax2 = axes[0, 1]
colores_dataset = {'KITTI': '#2196F3', '3D_Illusion': '#9C27B0'}
for dataset, grupo in df_total.groupby('dataset'):
    brd = grupo['densidad_bordes'].dropna()
    ax2.hist(brd, bins=20, alpha=0.6, label=dataset,
             color=colores_dataset.get(dataset, 'gray'), edgecolor='white')
ax2.axvline(UMBRAL_BORDES, color='#D32F2F', linewidth=2, linestyle='--',
            label=f'Mediana = {UMBRAL_BORDES:.4f}')
ax2.set_xlabel('Densidad de bordes normalizada')
ax2.set_ylabel('Número de imágenes')
ax2.set_title('Distribución de complejidad visual (distractores)')
ax2.legend()

# --- GRÁFICA 3: Balance factorial KITTI ---
ax3 = axes[1, 0]
kitti_balance = df_total[df_total['dataset'] == 'KITTI'].groupby(
    ['nivel_disparidad', 'presencia_distractores']
).size().reset_index(name='n')
etiquetas = [f"{r['nivel_disparidad'].upper()}\n{r['presencia_distractores'].replace('_', ' ')}"
             for _, r in kitti_balance.iterrows()]
colores_bal = ['#42A5F5', '#1565C0', '#FFA726', '#E65100']
bars = ax3.bar(range(len(kitti_balance)), kitti_balance['n'],
               color=colores_bal[:len(kitti_balance)], edgecolor='white')
ax3.set_xticks(range(len(kitti_balance)))
ax3.set_xticklabels(etiquetas, fontsize=9)
ax3.set_ylabel('Número de imágenes')
ax3.set_title('Balance factorial — KITTI')
ax3.axhline(25, color='red', linestyle=':', linewidth=1, label='Ideal = 25')
ax3.legend()
for bar, val in zip(bars, kitti_balance['n']):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             str(val), ha='center', va='bottom', fontsize=11, fontweight='bold')

# --- GRÁFICA 4: Balance factorial Ilusiones ---
ax4 = axes[1, 1]
ill_balance = df_total[df_total['dataset'] == '3D_Illusion'].groupby(
    ['nivel_disparidad', 'presencia_distractores']
).size().reset_index(name='n')
etiquetas_ill = [f"{r['nivel_disparidad'].upper()}\n{r['presencia_distractores'].replace('_', ' ')}"
                 for _, r in ill_balance.iterrows()]
colores_ill = ['#AB47BC', '#6A1B9A', '#EC407A', '#880E4F']
bars_ill = ax4.bar(range(len(ill_balance)), ill_balance['n'],
                   color=colores_ill[:len(ill_balance)], edgecolor='white')
ax4.set_xticks(range(len(ill_balance)))
ax4.set_xticklabels(etiquetas_ill, fontsize=9)
ax4.set_ylabel('Número de imágenes')
ax4.set_title('Balance factorial — Ilusiones Visuales')
ax4.axhline(25, color='red', linestyle=':', linewidth=1, label='Ideal = 25')
ax4.legend()
for bar, val in zip(bars_ill, ill_balance['n']):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             str(val), ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()

# Guardar figura
ruta_fig = os.path.join(RESULTS_DIR, 'factorial_distributions.png')
plt.savefig(ruta_fig, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figura guardada en: {ruta_fig}')

## Celda 9 — Guardar el CSV maestro de clasificación factorial

In [ ]:
# ============================================================
# CELDA 9: Guardar CSV maestro
# ============================================================

# Columnas finales del CSV maestro
columnas_finales = [
    'imagen_id',
    'dataset',
    'tipo_tarea',
    'nivel_disparidad',
    'presencia_distractores',
    'condicion_factorial',
    'disparidad_media',
    'densidad_bordes',
    'ruta_img_izq',
    'ruta_disp'
]

# Columna tipo_ilusion solo para ilusiones
if 'tipo_ilusion' in df_total.columns:
    columnas_finales.insert(3, 'tipo_ilusion')

df_final = df_total[columnas_finales].copy()

# Guardar CSV
ruta_csv = os.path.join(RESULTS_DIR, 'factorial_labels.csv')
df_final.to_csv(ruta_csv, index=False)

# También guardar en data/splits para referencia en notebooks siguientes
ruta_csv_splits = os.path.join(SPLITS_DIR, 'factorial_labels.csv')
df_final.to_csv(ruta_csv_splits, index=False)

print('CSV maestro guardado exitosamente.')
print(f'  Ruta principal: {ruta_csv}')
print(f'  Ruta en splits: {ruta_csv_splits}')
print()
print(f'Total de registros: {len(df_final)}')
print()
print('Vista previa del CSV:')
print(df_final.head(10).to_string(index=False))
print()
print('Distribución final de condiciones:')
print(df_final['condicion_factorial'].value_counts().to_string())

## Celda 10 — Resumen final y verificación de balance

In [ ]:
# ============================================================
# CELDA 10: Resumen final y verificación de balance
# ============================================================
print('=' * 65)
print('RESUMEN FINAL — CLASIFICACIÓN FACTORIAL')
print('=' * 65)
print()
print(f'Total imágenes clasificadas: {len(df_final)}')
print(f'  → KITTI (discriminación de profundidad): {len(df_final[df_final["dataset"]=="KITTI"])}')
print(f'  → 3D Illusion (ilusiones visuales):      {len(df_final[df_final["dataset"]=="3D_Illusion"])}')
print()
print(f'Umbral de disparidad utilizado: {UMBRAL_DISPARIDAD:.4f} píxeles')
print(f'Umbral de distractores utilizado: {UMBRAL_BORDES:.6f} (densidad Sobel norm.)')
print()
print('Tabla factorial 2×2×2:')
print('-' * 65)

tabla_resumen = df_final.groupby(
    ['tipo_tarea', 'nivel_disparidad', 'presencia_distractores']
).size().reset_index(name='N')
tabla_resumen.columns = ['Tipo de tarea', 'Disparidad', 'Distractores', 'N imágenes']
print(tabla_resumen.to_string(index=False))

print()
print('Verificación de balance:')
ns = tabla_resumen['N imágenes']
if ns.min() >= 20 and ns.max() <= 30:
    print('  ✓ Diseño balanceado (todas las celdas entre 20 y 30 imágenes)')
else:
    print('  ⚠ Algunas celdas tienen desequilibrio. Revisa la distribución.')
    print(f'    Mínimo: {ns.min()} | Máximo: {ns.max()} | Ideal: 25')
    print('    Esto se corregirá en el Notebook 09 al seleccionar el subconjunto final.')

print()
print('Próximo paso:')
print('  Notebook 09 — Definición de pares A/B y etiquetas ground truth')
print('  El archivo factorial_labels.csv es el insumo principal.')
print()
print('Archivos generados por este notebook:')
print(f'  1. {ruta_csv}')
print(f'  2. {ruta_csv_splits}')
print(f'  3. {ruta_fig}')